In [12]:
# ================================================
# CELDA 1 - Imports
# ================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob, io, requests

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

plt.rcParams['figure.figsize'] = (10,6)
sns.set(style="whitegrid")


In [13]:
# ================================================
# CELDA 2 - Cargar dataset
# ================================================

github_raw_url = ("https://raw.githubusercontent.com/"
                  "Dilan140402/Analitica-Datos/"
                  "61f05dd96a1f6e07a1721930ef267b11651f18c9/"
                  "data/citybike_lima.csv")

local_files = glob.glob("*citybike*.csv") + glob.glob("*city*.csv")

def load_citybike(url, local_files):
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            print("CSV cargado desde GitHub.")
            return pd.read_csv(io.StringIO(r.text))
    except:
        print("Fallo descarga. Probando archivos locales...")

    for f in local_files:
        try:
            print("Cargando local:", f)
            return pd.read_csv(f)
        except:
            pass
    raise FileNotFoundError("No se encontró archivo válido.")

df = load_citybike(github_raw_url, local_files)
df.head()


CSV cargado desde GitHub.


,scrape_timestamp,station_id,station_name,lat,lon,capacity,free_bikes,empty_slots,day_of_week,periodo_dia,weather_main,weather_desc,temp_C,wind_speed,clima_miraflores,temp_miraflores,in_miraflores
0,2025-10-01T11:34:22.005486-05:00,008a35afc6b4060be57b48bf90bec44c,18027 Ov. Julio Ramón Riveyro - Av. Pardo,-12.119013,-77.039928,14,8,6,Wednesday,mañana,NaN,NaN,19.0,NaN,Clima: pronóstico del tiempo,19.0,True
1,2025-10-01T11:34:22.005486-05:00,03367da30caea302b11c838d8b98df55,18009 Ca. Luis Schereiber Cdra. 2 (C.C. Aurora),-12.122125,-77.011506,16,8,8,Wednesday,mañana,NaN,NaN,19.0,NaN,Clima: pronóstico del tiempo,19.0,False
2,2025-10-01T11:34:22.005486-05:00,06dd87a8b87232577015b1c9a4ba08ed,18024 Ov. Bolognesi - Ca. Madrid,-12.123368,-77.035637,14,3,11,Wednesday,mañana,NaN,NaN,19.0,NaN,Clima: pronóstico del tiempo,19.0,True
3,2025-10-01T11:34:22.005486-05:00,0927eccbf04e2aadd179595c55c52bbe,18047 Malecón Cisneros - Ca. Trípoli,-12.125037,-77.037307,16,9,7,Wednesday,mañana,NaN,NaN,19.0,NaN,Clima: pronóstico del tiempo,19.0,True
4,2025-10-01T11:34:22.005486-05:00,0ac132eb1a147b7b23a753185cbebd1d,18026 Malecón de la Marina - Parque Grau,-12.118497,-77.045048,20,11,9,Wednesday,mañana,NaN,NaN,19.0,NaN,Clima: pronóstico del tiempo,19.0,True


In [14]:
# ================================================
# CELDA 3 - Procesamiento básico y cálculo de promedios diarios
# ================================================

# Convertir timestamp
df["scrape_timestamp"] = pd.to_datetime(df["scrape_timestamp"], errors="coerce")

# Crear columna fecha
df["date"] = df["scrape_timestamp"].dt.date

# Limpiar datos
df_clean = df.dropna(subset=["date", "free_bikes"])

# Promedio diario por estación
promedio_diario = (
    df_clean.groupby(["station_name", "date"])["free_bikes"]
    .mean()
    .reset_index()
    .sort_values(["station_name", "date"])
)

print("Promedio diario de bicicletas por estación:")
display(promedio_diario.head())


Promedio diario de bicicletas por estación:


,station_name,date,free_bikes
0,18001 Ca. Oscar R. Benavides - Ca. Schell (Par...,2025-10-01,11.217391
1,18001 Ca. Oscar R. Benavides - Ca. Schell (Par...,2025-10-02,12.906977
2,18001 Ca. Oscar R. Benavides - Ca. Schell (Par...,2025-10-03,10.512195
3,18001 Ca. Oscar R. Benavides - Ca. Schell (Par...,2025-10-04,11.214286
4,18001 Ca. Oscar R. Benavides - Ca. Schell (Par...,2025-10-05,5.261905


In [ ]:
# ================================================
# CELDA 4 - Gráficos
# ================================================

stations = promedio_diario["station_name"].unique()
print("Total de estaciones:", len(stations))

for station in stations:
    subset = promedio_diario[promedio_diario["station_name"] == station]

    plt.figure(figsize=(12,4))
    plt.plot(subset["date"], subset["free_bikes"], marker="o", linewidth=2)

    plt.title(f"Promedio diario de bicicletas disponibles - {station}")
    plt.xlabel("Fecha")
    plt.ylabel("Promedio de bicicletas disponibles")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [28]:
# ================================================
# CELDA 5 - Limpieza avanzada y creación de features
# ================================================

# Normalizar nombres de columnas
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Convertir timestamp
df['scrape_timestamp'] = pd.to_datetime(df['scrape_timestamp'], errors='coerce')

# Nuevas columnas útiles
df['hour'] = df['scrape_timestamp'].dt.hour
df['weekday'] = df['scrape_timestamp'].dt.weekday

# Convertir booleanos (solo si aplica)
if 'in_miraflores' in df.columns:
    df['in_miraflores'] = df['in_miraflores'].astype(str).str.lower().map({
        'true': True, '1': True, 't': True,
        'false': False, '0': False, 'f': False
    })

print("Nulos por columna:")
print(df.isnull().sum())

df.head()


Nulos por columna:
scrape_timestamp         0
station_id               0
station_name             0
lat                      0
lon                      0
capacity                 0
free_bikes               0
empty_slots              0
day_of_week              0
periodo_dia              0
weather_main        113138
weather_desc        113138
temp_c                1850
wind_speed          113138
clima_miraflores      1850
temp_miraflores       1850
in_miraflores            0
date                     0
hour                     0
weekday                  0
dtype: int64


,scrape_timestamp,station_id,station_name,lat,lon,capacity,free_bikes,empty_slots,day_of_week,periodo_dia,weather_main,weather_desc,temp_c,wind_speed,clima_miraflores,temp_miraflores,in_miraflores,date,hour,weekday
0,2025-10-01 11:34:22.005486-05:00,008a35afc6b4060be57b48bf90bec44c,18027 Ov. Julio Ramón Riveyro - Av. Pardo,-12.119013,-77.039928,14,8,6,Wednesday,mañana,NaN,NaN,19.0,NaN,Clima: pronóstico del tiempo,19.0,True,2025-10-01,11,2
1,2025-10-01 11:34:22.005486-05:00,03367da30caea302b11c838d8b98df55,18009 Ca. Luis Schereiber Cdra. 2 (C.C. Aurora),-12.122125,-77.011506,16,8,8,Wednesday,mañana,NaN,NaN,19.0,NaN,Clima: pronóstico del tiempo,19.0,False,2025-10-01,11,2
2,2025-10-01 11:34:22.005486-05:00,06dd87a8b87232577015b1c9a4ba08ed,18024 Ov. Bolognesi - Ca. Madrid,-12.123368,-77.035637,14,3,11,Wednesday,mañana,NaN,NaN,19.0,NaN,Clima: pronóstico del tiempo,19.0,True,2025-10-01,11,2
3,2025-10-01 11:34:22.005486-05:00,0927eccbf04e2aadd179595c55c52bbe,18047 Malecón Cisneros - Ca. Trípoli,-12.125037,-77.037307,16,9,7,Wednesday,mañana,NaN,NaN,19.0,NaN,Clima: pronóstico del tiempo,19.0,True,2025-10-01,11,2
4,2025-10-01 11:34:22.005486-05:00,0ac132eb1a147b7b23a753185cbebd1d,18026 Malecón de la Marina - Parque Grau,-12.118497,-77.045048,20,11,9,Wednesday,mañana,NaN,NaN,19.0,NaN,Clima: pronóstico del tiempo,19.0,True,2025-10-01,11,2


In [37]:
# ======================================================
# CELDA FINAL — DETECCIÓN, FILTRADO E IMPUTACIÓN SEGURA
# ======================================================

from sklearn.impute import SimpleImputer

# 1) Normalizar nombres de columnas
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# 1.1 ELIMINAR columnas duplicadas
df = df.loc[:, ~df.columns.duplicated()]
print("✔ Columnas únicas aseguradas")

target = "free_bikes"
if target not in df.columns:
    raise ValueError("No encontré la columna free_bikes en el DataFrame")

# 2) Convertir columnas object a número si tiene sentido
for col in df.columns:
    if df[col].dtype == "object":
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().mean() > 0.7:  # si más del 70% puede ser número
            df[col] = converted

# 3) Detectar columnas numéricas reales
num_features = [
    col for col in df.columns
    if pd.api.types.is_numeric_dtype(df[col]) and col != target
]

# 3.1 Eliminar columnas numéricas completamente vacías
empty_num_cols = [col for col in num_features if df[col].isna().sum() == len(df)]
if empty_num_cols:
    print("⚠ Columnas numéricas eliminadas por estar vacías:", empty_num_cols)
    num_features = [col for col in num_features if col not in empty_num_cols]

# 4) Detectar columnas categóricas reales
cat_features = [
    col for col in df.columns
    if df[col].dtype in ["object", "bool"]
]

print("✔ Num features finales:", num_features)
print("✔ Cat features finales:", cat_features)

# 5) Crear df_model limpio con columnas únicas
use_cols = list(dict.fromkeys(num_features + cat_features + [target]))
df_model = df[use_cols].copy()

# 6) Crear imputadores
num_imputer = SimpleImputer(strategy="mean")
cat_imputer = SimpleImputer(strategy="constant", fill_value="__missing__")

# 7) Imputación segura (sin columnas fantasma)
df_model[num_features] = pd.DataFrame(
    num_imputer.fit_transform(df_model[num_features]),
    columns=num_features,
    index=df_model.index
)

df_model[cat_features] = pd.DataFrame(
    cat_imputer.fit_transform(df_model[cat_features]),
    columns=cat_features,
    index=df_model.index
)

print("\n🎉 Imputación completada sin errores")
df_model.head()


✔ Columnas únicas aseguradas
⚠ Columnas numéricas eliminadas por estar vacías: ['weather_main', 'weather_desc', 'wind_speed']
✔ Num features finales: ['lat', 'lon', 'capacity', 'empty_slots', 'temp_c', 'temp_miraflores', 'in_miraflores', 'hour', 'weekday']
✔ Cat features finales: ['station_id', 'station_name', 'day_of_week', 'periodo_dia', 'clima_miraflores', 'in_miraflores', 'date']

🎉 Imputación completada sin errores


,lat,lon,capacity,empty_slots,temp_c,temp_miraflores,in_miraflores,hour,weekday,station_id,station_name,day_of_week,periodo_dia,clima_miraflores,date,free_bikes
0,-12.119013,-77.039928,14.0,6.0,19.0,19.0,1.0,11.0,2.0,008a35afc6b4060be57b48bf90bec44c,18027 Ov. Julio Ramón Riveyro - Av. Pardo,Wednesday,mañana,Clima: pronóstico del tiempo,2025-10-01,8
1,-12.122125,-77.011506,16.0,8.0,19.0,19.0,0.0,11.0,2.0,03367da30caea302b11c838d8b98df55,18009 Ca. Luis Schereiber Cdra. 2 (C.C. Aurora),Wednesday,mañana,Clima: pronóstico del tiempo,2025-10-01,8
2,-12.123368,-77.035637,14.0,11.0,19.0,19.0,1.0,11.0,2.0,06dd87a8b87232577015b1c9a4ba08ed,18024 Ov. Bolognesi - Ca. Madrid,Wednesday,mañana,Clima: pronóstico del tiempo,2025-10-01,3
3,-12.125037,-77.037307,16.0,7.0,19.0,19.0,1.0,11.0,2.0,0927eccbf04e2aadd179595c55c52bbe,18047 Malecón Cisneros - Ca. Trípoli,Wednesday,mañana,Clima: pronóstico del tiempo,2025-10-01,9
4,-12.118497,-77.045048,20.0,9.0,19.0,19.0,1.0,11.0,2.0,0ac132eb1a147b7b23a753185cbebd1d,18026 Malecón de la Marina - Parque Grau,Wednesday,mañana,Clima: pronóstico del tiempo,2025-10-01,11


In [34]:
print("num_features:", num_features)
print("Cantidad num_features:", len(num_features))
print("Shape antes de imputar:", df_model[num_features].shape)

arr = num_imputer.fit_transform(df_model[num_features])

print("Shape imputado:", arr.shape)


num_features: ['lat', 'lon', 'capacity', 'empty_slots', 'temp_c', 'temp_miraflores', 'in_miraflores', 'hour', 'weekday']
Cantidad num_features: 9
Shape antes de imputar: (113138, 10)
Shape imputado: (113138, 10)


In [35]:
print("Columnas reales en df_model[num_features]:")
print(df_model[num_features].columns.tolist())

Columnas reales en df_model[num_features]:
['lat', 'lon', 'capacity', 'empty_slots', 'temp_c', 'temp_miraflores', 'in_miraflores', 'in_miraflores', 'hour', 'weekday']


In [36]:
print("Todas las columnas de df_model:")
print(df_model.columns.tolist())
print("Cantidad:", len(df_model.columns))

Todas las columnas de df_model:
['lat', 'lon', 'capacity', 'empty_slots', 'temp_c', 'temp_miraflores', 'in_miraflores', 'hour', 'weekday', 'station_id', 'station_name', 'day_of_week', 'periodo_dia', 'clima_miraflores', 'in_miraflores', 'date', 'free_bikes']
Cantidad: 17


In [39]:
# ================================================
# CELDA 4 - Selección de variables y manejo de NaN
# ================================================

# 0) Normalizar columnas y ELIMINAR duplicadas
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df.loc[:, ~df.columns.duplicated()]

print("✔ Columnas únicas aseguradas:", len(df.columns))

target = "free_bikes"
if target not in df.columns:
    raise ValueError("No encontré la columna free_bikes en el DataFrame")

# ------------------------------------------------------
# 1. Intentar convertir objetos a número (solo si tiene sentido)
# ------------------------------------------------------
for col in df.columns:
    if df[col].dtype == "object":
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().mean() > 0.9:  # si 90% es convertible → numérica real
            df[col] = converted

# ------------------------------------------------------
# 2. Detectar correctamente columnas numéricas
# ------------------------------------------------------
num_features = [
    col for col in df.columns
    if pd.api.types.is_numeric_dtype(df[col]) and col != target
]

# Eliminar columnas numéricas completamente vacías
num_features = [
    col for col in num_features
    if df[col].isna().sum() < len(df)
]

# Detectar categóricas
cat_features = [
    col for col in df.columns
    if df[col].dtype == "object" or df[col].dtype == "bool"
]

print("\n✔ Num features finales:", num_features)
print("✔ Cat features finales:", cat_features)

# ------------------------------------------------------
# 3. Crear df_model solo con columnas válidas
# ------------------------------------------------------
use_cols = num_features + cat_features + [target]

# Evitar duplicados aquí también
use_cols = list(dict.fromkeys(use_cols))

df_model = df[use_cols].copy()

# ------------------------------------------------------
# 4. IMPUTACIÓN SEGURA
# ------------------------------------------------------
num_imputer = SimpleImputer(strategy="mean")
cat_imputer = SimpleImputer(strategy="constant", fill_value="__missing__")

df_model[num_features] = pd.DataFrame(
    num_imputer.fit_transform(df_model[num_features]),
    columns=num_features,
    index=df_model.index
)

df_model[cat_features] = pd.DataFrame(
    cat_imputer.fit_transform(df_model[cat_features]),
    columns=cat_features,
    index=df_model.index
)

print("\n🎉 Imputación completada sin errores")
df_model.head()


✔ Columnas únicas aseguradas: 20

✔ Num features finales: ['lat', 'lon', 'capacity', 'empty_slots', 'temp_c', 'temp_miraflores', 'in_miraflores', 'hour', 'weekday']
✔ Cat features finales: ['station_id', 'station_name', 'day_of_week', 'periodo_dia', 'clima_miraflores', 'in_miraflores', 'date']

🎉 Imputación completada sin errores


,lat,lon,capacity,empty_slots,temp_c,temp_miraflores,in_miraflores,hour,weekday,station_id,station_name,day_of_week,periodo_dia,clima_miraflores,date,free_bikes
0,-12.119013,-77.039928,14.0,6.0,19.0,19.0,1.0,11.0,2.0,008a35afc6b4060be57b48bf90bec44c,18027 Ov. Julio Ramón Riveyro - Av. Pardo,Wednesday,mañana,Clima: pronóstico del tiempo,2025-10-01,8
1,-12.122125,-77.011506,16.0,8.0,19.0,19.0,0.0,11.0,2.0,03367da30caea302b11c838d8b98df55,18009 Ca. Luis Schereiber Cdra. 2 (C.C. Aurora),Wednesday,mañana,Clima: pronóstico del tiempo,2025-10-01,8
2,-12.123368,-77.035637,14.0,11.0,19.0,19.0,1.0,11.0,2.0,06dd87a8b87232577015b1c9a4ba08ed,18024 Ov. Bolognesi - Ca. Madrid,Wednesday,mañana,Clima: pronóstico del tiempo,2025-10-01,3
3,-12.125037,-77.037307,16.0,7.0,19.0,19.0,1.0,11.0,2.0,0927eccbf04e2aadd179595c55c52bbe,18047 Malecón Cisneros - Ca. Trípoli,Wednesday,mañana,Clima: pronóstico del tiempo,2025-10-01,9
4,-12.118497,-77.045048,20.0,9.0,19.0,19.0,1.0,11.0,2.0,0ac132eb1a147b7b23a753185cbebd1d,18026 Malecón de la Marina - Parque Grau,Wednesday,mañana,Clima: pronóstico del tiempo,2025-10-01,11


In [41]:
# CELDA 5 - Train/test y preprocesador
X = df_model[num_features + cat_features]
y = df_model[target].astype(float)

print("Filas disponibles para modelar:", X.shape[0])
if X.shape[0] < 3:
    print("Advertencia: muy pocas filas. Algunos modelos pueden no generalizar.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer([
    ("num", "passthrough", num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features)
], remainder="drop")

Filas disponibles para modelar: 113138


In [43]:
# ================================
# CELDA 6 - Transformación segura
# ================================

# 1) Eliminar columnas duplicadas en X_train / X_test
X_train = X_train.loc[:, ~X_train.columns.duplicated()]
X_test = X_test.loc[:, ~X_test.columns.duplicated()]

# 2) Asegurar que las listas de columnas no tengan duplicados
num_features = list(dict.fromkeys(num_features))
cat_features = list(dict.fromkeys(cat_features))

# 3) Reconstruir el preprocessor limpio
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ],
    remainder="drop"
)

# 4) Transformar
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# 5) Asegurar que y sea vector columna
y_train_arr = y_train.values.reshape(-1, 1)
y_test_arr = y_test.values.reshape(-1, 1)

print("X_train_proc shape:", X_train_proc.shape)
print("X_test_proc shape:", X_test_proc.shape)
print("y_train shape:", y_train_arr.shape)


X_train_proc shape: (90510, 185)
X_test_proc shape: (22628, 185)
y_train shape: (90510, 1)


In [51]:
# --- ELIMINAR COLUMNAS DUPLICADAS DE X ---

print("Columnas originales:", len(X.columns))
print("Duplicadas:", X.columns[X.columns.duplicated()].tolist())

# quitar duplicadas conservando la primera aparición
X = X.loc[:, ~X.columns.duplicated()]

print("Columnas después de limpiar:", len(X.columns))


Columnas originales: 16
Duplicadas: ['in_miraflores']
Columnas después de limpiar: 15


In [57]:
# --- RECONSTRUIR PROCESAMIENTO ---

# REHACER split (para estar seguros que X_train tiene las filas correctas)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# aplicar preprocessor correctamente
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

# Asegurar forma correcta
print("X_train_proc shape:", X_train_proc.shape)
print("X_test_proc shape:", X_test_proc.shape)


X_train_proc shape: (90510, 185)
X_test_proc shape: (22628, 185)


In [63]:
# CELDA 7 - Ecuación Normal
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# reconstruir matrices
Xb = np.c_[np.ones((X_train_proc.shape[0], 1)), X_train_proc]
Xb_test = np.c_[np.ones((X_test_proc.shape[0], 1)), X_test_proc]

try:
    theta_inv = np.linalg.inv(Xb.T @ Xb) @ (Xb.T @ y_train_arr)
    print("Ecuación normal via inversa calculada.")
except np.linalg.LinAlgError:
    theta_inv = np.linalg.pinv(Xb) @ y_train_arr
    print("Matriz singular → usando pseudoinversa.")

y_pred_normal = (Xb_test @ theta_inv).flatten()

rmse_normal = np.sqrt(mean_squared_error(y_test_arr, y_pred_normal))
r2_normal = r2_score(y_test_arr, y_pred_normal)
mae_normal = mean_absolute_error(y_test_arr, y_pred_normal)

print(f"Ecuación Normal → RMSE: {rmse_normal:.4f}, R2: {r2_normal:.4f}, MAE: {mae_normal:.4f}")



Ecuación normal via inversa calculada.
Ecuación Normal → RMSE: 6429277709576799232.0000, R2: -2295044920469789351118949238007922688.0000, MAE: 5134877395042882560.0000


In [64]:
# CELDA 8 - Comparar con pinv y lstsq

# reconfirmar matrices
Xb = np.c_[np.ones((X_train_proc.shape[0], 1)), X_train_proc]
Xb_test = np.c_[np.ones((X_test_proc.shape[0], 1)), X_test_proc]

# pseudoinversa
theta_pinv = np.linalg.pinv(Xb) @ y_train_arr
y_pred_pinv = (Xb_test @ theta_pinv).flatten()
rmse_pinv = np.sqrt(mean_squared_error(y_test_arr, y_pred_pinv))

# lstsq (SVD)
theta_lstsq, _, _, _ = np.linalg.lstsq(Xb, y_train_arr, rcond=None)
y_pred_lstsq = (Xb_test @ theta_lstsq).flatten()
rmse_lstsq = np.sqrt(mean_squared_error(y_test_arr, y_pred_lstsq))

print(f"pinv RMSE: {rmse_pinv:.4f}")
print(f"lstsq RMSE: {rmse_lstsq:.4f}")


pinv RMSE: 0.0837
lstsq RMSE: 0.0837


In [65]:
# CELDA 9 - LinearRegression con Pipeline

pipeline_lr = Pipeline([
    ("pre", preprocessor),
    ("lr", LinearRegression())
])

pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print(f"LinearRegression → RMSE: {rmse_lr:.4f}, R2: {r2_lr:.4f}, MAE: {mae_lr:.4f}")


LinearRegression → RMSE: 0.0837, R2: 0.9996, MAE: 0.0244


In [66]:
# CELDA 11 - Coeficientes: para theta_inv/pinv/lstsq y coef. sklearn

# nombres de columnas tras preprocesado
# Obtener nombres de features del preprocessor (solo para OneHotEncoder case)
num_names = num_features
cat_encoder = preprocessor.named_transformers_["cat"]
if hasattr(cat_encoder, "get_feature_names_out"):
    cat_names = list(cat_encoder.get_feature_names_out(cat_features))
else:
    cat_names = []
feature_names = ["bias"] + num_names + cat_names

def print_coef(theta, names, method_name):
    vec = theta.flatten()
    df_coef = pd.DataFrame({"feature": names, "coef": vec})
    display(df_coef.sort_values("coef", key=lambda s: s.abs(), ascending=False).head(20))
    df_coef.to_csv(f"graficos/{method_name}_coefs.csv", index=False)

print("Feature names (preprocesadas):", feature_names)

# Para theta_inv (usamos theta_inv calculado arriba)
print_coef(theta_inv, feature_names, "EcuacionNormal")

# Para sklearn
sk_coef = pipeline_lr.named_steps["lr"].coef_.flatten()
sk_intercept = pipeline_lr.named_steps["lr"].intercept_
sk_feature_names = num_names + (list(pipeline_lr.named_steps["pre"].named_transformers_["cat"].get_feature_names_out(cat_features)) if hasattr(pipeline_lr.named_steps["pre"].named_transformers_["cat"], "get_feature_names_out") else cat_features)
df_sk = pd.DataFrame({"feature": ["bias"]+sk_feature_names, "coef": np.concatenate([sk_intercept.reshape(-1), sk_coef])})
display(df_sk)
df_sk.to_csv("graficos/sklearn_coefs.csv", index=False)


Feature names (preprocesadas): ['bias', 'lat', 'lon', 'capacity', 'empty_slots', 'temp_c', 'temp_miraflores', 'in_miraflores', 'hour', 'weekday', 'station_id_008a35afc6b4060be57b48bf90bec44c', 'station_id_03367da30caea302b11c838d8b98df55', 'station_id_06dd87a8b87232577015b1c9a4ba08ed', 'station_id_0927eccbf04e2aadd179595c55c52bbe', 'station_id_0ac132eb1a147b7b23a753185cbebd1d', 'station_id_11c5d99df877943158c7e7c693242e26', 'station_id_1ff7b00c8a2af2eeb129df70939eb20b', 'station_id_263e9a1c4f245c4070f8c85b88d0f10d', 'station_id_3ab2451c995f47035d2b14590753236a', 'station_id_3c8e489aaccbcbbe87f59457fe1901d1', 'station_id_409c57fe8f27975938e2e0248a4d45bc', 'station_id_4158a25459abe328f8ab8c32c0db7b6b', 'station_id_438d63a7c5cb69db70e9cd6665120d61', 'station_id_439e3845a612d6b0a61506b0ebe52a91', 'station_id_55c0bf37006a2c06875866c6e9e13156', 'station_id_57ded81c72e11e6fc644698ee200d750', 'station_id_5b7914e3a19e2d392967c8ff9a1eeb5e', 'station_id_6369e20f974a82f599ac39793a9ceeef', 'station

,feature,coef
73,station_name_18011 Ca. GermánA.Goméz - Av.Jose...,-2.152045e+34
53,station_id_c00d815ca6322144c5510d9f9e7121da,2.152045e+34
6,temp_miraflores,8.800410e+31
5,temp_c,-8.800410e+31
77,station_name_18015 Ca. Bolivar - Ca. Alcanfores,-9.505447e+23
56,station_id_dc710115aa6014501e5b457a0b14ef25,9.503002e+23
74,station_name_18012 Ca. Ignacio La Puente - Av....,-3.350504e+23
61,station_id_eff5bc44fad7421819820f56aa6ded68,3.347969e+23
113,station_name_27042 Av. P. Carriquiry - Av. Ric...,-2.702686e+23
27,station_id_6369e20f974a82f599ac39793a9ceeef,2.686121e+23


,feature,coef
0,bias,-0.024984
1,lat,-0.000290
2,lon,-0.000797
3,capacity,0.996874
4,empty_slots,-0.999537
...,...,...
181,date_2025-11-23,-0.036309
182,date_2025-11-24,-0.033355
183,date_2025-11-25,-0.046776
184,date_2025-11-26,-0.016156


In [67]:
# CELDA 12 - Resumen de métricas en tabla
from sklearn.metrics import mean_absolute_error
results = []
results.append(["EcuacionNormal", rmse_normal, r2_normal, mean_absolute_error(y_test_arr, y_pred_normal)])
results.append(["Pseudoinversa", rmse_pinv, r2_score(y_test_arr, y_pred_pinv), mean_absolute_error(y_test_arr, y_pred_pinv)])
results.append(["Lstsq", rmse_lstsq, r2_score(y_test_arr, y_pred_lstsq), mean_absolute_error(y_test_arr, y_pred_lstsq)])
results.append(["LinearRegression", rmse_lr, r2_lr, mae_lr])

res_df = pd.DataFrame(results, columns=["model","rmse","r2","mae"]).sort_values("rmse")
display(res_df)
res_df.to_csv("graficos/metrics_comparison.csv", index=False)


,model,rmse,r2,mae
1,Pseudoinversa,8.369282e-02,9.996111e-01,2.442389e-02
2,Lstsq,8.369282e-02,9.996111e-01,2.442389e-02
3,LinearRegression,8.369292e-02,9.996111e-01,2.442369e-02
0,EcuacionNormal,6.429278e+18,-2.295045e+36,5.134877e+18
